# 10 — Public Population Warehouse + Forensics (CPU, v4.2 LOW-RAM)

This version fixes the Kaggle kernel death seen immediately after input discovery.

### Why v4.1 died

The schema-audit cell opened every table as a full pandas DataFrame. That is harmless for
`episodes.csv`, but catastrophic for a large `replays.parquet`: the kernel can be OOM-killed
before Python gets a chance to raise an exception.

### v4.2 memory rules

- `episodes.csv` is the **only** large research table intentionally read in full here.
- Parquet files are inspected through **PyArrow metadata only**.
- CSV schemas are inspected with `nrows=5`.
- Large CSV counts use `wc -l`, which streams from disk and does not allocate the table.
- `stream_hashes.csv` is reduced in chunks.
- `daily_stats.csv` and `episode_features.csv` are sampled only. Full aggregation happens in notebook 11.
- `replays.parquet` is **never** materialized in notebook 10.
- The official Episodes Index is inspected lightly rather than copied into RAM.

## Kaggle inputs

1. **Kaggriculture Episodes** public bundle (`episodes.csv`, `daily_stats.csv`, `replays.parquet`, ...)
2. **Kaggriculture Episodes Index**
3. `kaggriculture-code-repo` is optional for this notebook

**Accelerator:** None / CPU  
**Internet:** OFF


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, gc, math, collections
import pandas as pd
import numpy as np

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/kagv2')
WORK.mkdir(parents=True, exist_ok=True)

def find_repo():
    if not INPUT.exists():
        return None
    for p in INPUT.rglob('__init__.py'):
        if p.parent.name == 'kagv2' and p.parent.parent.name == 'src':
            return p.parents[2]
    return None

ROOT = find_repo()
if ROOT:
    sys.path.insert(0, str(ROOT))
    sys.path.insert(0, str(ROOT/'src'))

print('ROOT =', ROOT or 'not required')
print('WORK =', WORK)
print('CPU_COUNT =', os.cpu_count())


In [ ]:
# ---------- Self-contained recursive input discovery ----------
def files_named(root, name):
    root = Path(root)
    return sorted(p for p in root.rglob(name) if p.is_file()) if root.exists() else []

def find_bundle():
    for p in files_named(INPUT, 'episodes.csv'):
        parent = p.parent
        if (parent/'replays.parquet').exists():
            return parent
    return None

def find_index():
    hits = files_named(INPUT, 'manifest.csv')
    preferred = [p.parent for p in hits
                 if 'kaggriculture' in str(p).lower() and 'episode' in str(p).lower()]
    return preferred[0] if preferred else (hits[0].parent if len(hits)==1 else None)

PUBLIC_ROOT = find_bundle()
INDEX_ROOT = find_index()

found = {
    'public_episode_bundle': str(PUBLIC_ROOT) if PUBLIC_ROOT else None,
    'official_episode_index': str(INDEX_ROOT) if INDEX_ROOT else None,
    'code_repo': str(ROOT) if ROOT else None,
}
print(json.dumps(found, indent=2))
(WORK/'input_discovery.json').write_text(json.dumps(found, indent=2))

if PUBLIC_ROOT is None:
    raise FileNotFoundError(
        'Could not locate a directory containing both episodes.csv and replays.parquet.'
    )


In [ ]:
# ---------- LOW-RAM schema audit ----------
import pyarrow.parquet as pq

KNOWN_PUBLIC = [
    'episodes.csv',
    'episode_features.csv',
    'daily_stats.csv',
    'teams.csv',
    'stream_hashes.csv',
    'replays.parquet',
]
KNOWN_INDEX = ['manifest.csv']

def wc_rows(path):
    try:
        r = subprocess.run(['wc','-l',str(path)], capture_output=True, text=True, timeout=120)
        if r.returncode == 0:
            return max(0, int(r.stdout.split()[0]) - 1)
    except Exception:
        pass
    return None

def audit_one(path, source):
    p = Path(path)
    row = {
        'source': source,
        'path': str(p),
        'size_mb': round(p.stat().st_size / 2**20, 3),
        'suffix': p.suffix.lower(),
    }
    try:
        if p.suffix.lower() == '.parquet':
            pf = pq.ParquetFile(p)
            row['rows'] = int(pf.metadata.num_rows)
            row['cols'] = len(pf.schema_arrow.names)
            row['row_groups'] = int(pf.metadata.num_row_groups)
            row['columns'] = ' | '.join(pf.schema_arrow.names[:100])
            del pf
        elif p.suffix.lower() == '.csv':
            sample = pd.read_csv(p, nrows=5)
            row['rows'] = wc_rows(p)
            row['cols'] = len(sample.columns)
            row['columns'] = ' | '.join(map(str, sample.columns[:100]))
            del sample
        else:
            row['rows'] = None
            row['cols'] = None
            row['columns'] = ''
    except Exception as e:
        row['error'] = repr(e)
    return row

rows = []
for name in KNOWN_PUBLIC:
    p = PUBLIC_ROOT/name
    if p.exists():
        rows.append(audit_one(p, 'public_bundle'))
if INDEX_ROOT:
    for name in KNOWN_INDEX:
        p = INDEX_ROOT/name
        if p.exists():
            rows.append(audit_one(p, 'official_index'))

schema = pd.DataFrame(rows)
schema.to_csv(WORK/'public_corpus_schema.csv', index=False)
display(schema)
gc.collect()


In [ ]:
# ---------- Load ONLY episodes.csv fully ----------
episodes_path = PUBLIC_ROOT/'episodes.csv'
episodes = pd.read_csv(episodes_path, low_memory=False)

if 'created_at' not in episodes.columns:
    for c in ['create_time','createTime','createtime']:
        if c in episodes.columns:
            episodes['created_at'] = pd.to_datetime(episodes[c], errors='coerce', utc=True)
            break
else:
    episodes['created_at'] = pd.to_datetime(episodes['created_at'], errors='coerce', utc=True)

for c in ['bank_0','bank_1','rating_0','rating_1']:
    if c in episodes.columns:
        episodes[c] = pd.to_numeric(episodes[c], errors='coerce')

if {'bank_0','bank_1'}.issubset(episodes.columns):
    episodes['margin_0'] = episodes['bank_0'] - episodes['bank_1']
    episodes['winner'] = np.where(episodes.bank_0 > episodes.bank_1, 1.0, np.where(episodes.bank_0 < episodes.bank_1, 0.0, 0.5))

if {'rating_0','rating_1'}.issubset(episodes.columns):
    episodes['rating_diff_0'] = episodes.rating_0 - episodes.rating_1
    episodes['rating_mean'] = (episodes.rating_0 + episodes.rating_1) / 2

if 'created_at' in episodes.columns:
    episodes['current_engine'] = episodes.created_at >= pd.Timestamp('2026-08-07', tz='UTC')
else:
    episodes['current_engine'] = False

episodes.to_parquet(WORK/'episodes_master.parquet', index=False)
print('episodes:', len(episodes))
if 'created_at' in episodes:
    print('date range:', episodes.created_at.min(), '->', episodes.created_at.max())
    print('post-2026-08-07:', int(episodes.current_engine.sum()))
display(episodes.head())


In [ ]:
# ---------- One-row-per-player table ----------
def wide_to_players(d):
    rows=[]
    common=[c for c in ['episode_id','created_at','create_time','state','type','current_engine'] if c in d.columns]
    for seat in (0,1):
        z=d[common].copy()
        z['seat']=seat
        mapping={f'sub_{seat}':'submission_id',f'team_{seat}':'team_id',f'bank_{seat}':'reward',f'rating_{seat}':'rating',f'sub_{1-seat}':'opponent_submission_id',f'team_{1-seat}':'opponent_team_id',f'bank_{1-seat}':'opponent_reward',f'rating_{1-seat}':'opponent_rating'}
        for src,dst in mapping.items():
            if src in d:
                z[dst]=d[src].to_numpy()
        if {'reward','opponent_reward'}.issubset(z.columns):
            z['win_target']=np.where(z.reward>z.opponent_reward,1.0,np.where(z.reward<z.opponent_reward,0.0,0.5))
            z['margin']=z.reward-z.opponent_reward
        rows.append(z)
    return pd.concat(rows, ignore_index=True)

players = wide_to_players(episodes)
players.to_parquet(WORK/'episode_players.parquet', index=False)
print('player rows:', len(players))
display(players.head())


In [ ]:
# ---------- Lightweight Bradley–Terry on current-era games ----------
def bradley_terry(match, iterations=250, lr=.03, l2=.01):
    actors = sorted(set(match.actor_a.astype(str)) | set(match.actor_b.astype(str)))
    idx = {a:i for i,a in enumerate(actors)}
    s = np.zeros(len(actors), dtype=float)
    counts = np.zeros(len(actors), dtype=int)
    ia = match.actor_a.astype(str).map(idx).to_numpy(); ib = match.actor_b.astype(str).map(idx).to_numpy(); y = match.y.to_numpy(float)
    for i in ia: counts[i]+=1
    for i in ib: counts[i]+=1
    for _ in range(iterations):
        grad = -l2*s
        d = np.clip(s[ia]-s[ib], -30, 30)
        p = 1/(1+np.exp(-d)); e = y-p
        np.add.at(grad, ia, e); np.add.at(grad, ib, -e)
        s += lr*grad/max(1,len(match)); s -= s.mean()
    return pd.DataFrame({'actor':actors,'strength':s,'games':counts}).sort_values(['strength','games'],ascending=False).reset_index(drop=True)

bt = pd.DataFrame(); match = pd.DataFrame()
if {'sub_0','sub_1','bank_0','bank_1'}.issubset(episodes.columns):
    m = episodes.dropna(subset=['sub_0','sub_1','bank_0','bank_1']).copy()
    if 'current_engine' in m and m.current_engine.any(): m = m[m.current_engine].copy()
    match = pd.DataFrame({'actor_a':m.sub_0.astype(str),'actor_b':m.sub_1.astype(str),'reward_a':m.bank_0.astype(float),'reward_b':m.bank_1.astype(float)})
    match['y'] = np.where(match.reward_a>match.reward_b,1.0,np.where(match.reward_a<match.reward_b,0.0,0.5))
    match.to_parquet(WORK/'matchups.parquet',index=False)
    bt = bradley_terry(match)
    bt.to_csv(WORK/'bt_strength.csv',index=False)
    print('BT matches:',len(match),'submissions:',len(bt))
    display(bt.head(30))


In [ ]:
# ---------- Chunked stream-hash reduction ----------
hash_path = PUBLIC_ROOT/'stream_hashes.csv'; hash_meta = {}
if hash_path.exists():
    head = pd.read_csv(hash_path, nrows=20)
    hash_cols=[c for c in head.columns if 'hash' in str(c).lower()]
    sub_cols=[c for c in head.columns if 'sub' in str(c).lower() or 'submission' in str(c).lower()]
    print('stream_hash columns:', list(head.columns)); print('candidate hash columns:',hash_cols); print('candidate submission columns:',sub_cols)
    if hash_cols:
        hcol=hash_cols[0]; counter=collections.Counter(); total=0
        for chunk in pd.read_csv(hash_path, usecols=[hcol], chunksize=200_000):
            vc=chunk[hcol].dropna().astype(str).value_counts(); counter.update(vc.to_dict()); total += len(chunk); del chunk, vc
        freq=pd.DataFrame(counter.items(),columns=[hcol,'rows']).sort_values('rows',ascending=False)
        freq.to_csv(WORK/'strategy_hash_frequency.csv',index=False)
        hash_meta={'rows_processed':total,'unique_hashes':len(freq),'hash_column':hcol}
        display(freq.head(30))
    del head; gc.collect()
else: print('No stream_hashes.csv')


In [ ]:
# ---------- Sample large feature CSVs, never load them whole ----------
samples={}
for name in ['daily_stats.csv','episode_features.csv']:
    p=PUBLIC_ROOT/name
    if not p.exists(): continue
    sample=pd.read_csv(p,nrows=25_000,low_memory=False)
    sample.to_parquet(WORK/f'{name[:-4]}_sample.parquet',index=False)
    samples[name]={'rows_sampled':len(sample),'cols':len(sample.columns),'columns':list(map(str,sample.columns)),'numeric_columns':list(map(str,sample.select_dtypes(include=[np.number]).columns))}
    print(name,'sample',sample.shape); display(sample.head(3)); del sample; gc.collect()
(WORK/'large_table_samples.json').write_text(json.dumps(samples,indent=2))


In [ ]:
# ---------- Inspect replays.parquet with metadata ONLY ----------
rp=PUBLIC_ROOT/'replays.parquet'; replay_meta={}
if rp.exists():
    pf=pq.ParquetFile(rp)
    replay_meta={'path':str(rp),'size_mb':round(rp.stat().st_size/2**20,3),'rows':int(pf.metadata.num_rows),'row_groups':int(pf.metadata.num_row_groups),'columns':pf.schema_arrow.names,'schema':str(pf.schema_arrow)}
    (WORK/'replays_metadata.json').write_text(json.dumps(replay_meta,indent=2))
    pd.DataFrame({'column':pf.schema_arrow.names,'type':[str(pf.schema_arrow.field(i).type) for i in range(len(pf.schema_arrow.names))]}).to_csv(WORK/'replays_schema.csv',index=False)
    print(json.dumps({k:v for k,v in replay_meta.items() if k!='schema'},indent=2))
    del pf; gc.collect()
# IMPORTANT: no pd.read_parquet(replays.parquet) in this notebook.


In [ ]:
# ---------- Freshness + memory-safe warehouse summary ----------
public_max = str(episodes.created_at.max()) if 'created_at' in episodes else None
current_n = int(episodes.current_engine.sum()) if 'current_engine' in episodes else 0
summary={'public_root':str(PUBLIC_ROOT),'index_root':str(INDEX_ROOT) if INDEX_ROOT else None,'episodes_rows':int(len(episodes)),'public_min_date':str(episodes.created_at.min()) if 'created_at' in episodes else None,'public_max_date':public_max,'public_current_engine_rows':current_n,'bt_matches':int(len(match)),'bt_submissions':int(len(bt)),'hash_reduction':hash_meta,'replays_rows':replay_meta.get('rows'),'replays_size_mb':replay_meta.get('size_mb'),'large_tables_sampled':list(samples),'memory_policy':'metadata/chunk/sample; replays parquet never materialized'}
(WORK/'public_warehouse_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
if 'created_at' in episodes and episodes.created_at.max() < pd.Timestamp('2026-08-07',tz='UTC'):
    print('\nWARNING: public packed corpus appears entirely pre-Aug-7.')
    print('Use it for general strategy structure, but acquire a targeted current-engine replay set before final policy promotion.')
elif current_n < 100:
    print('\nWARNING: very few current-engine episodes in the packed public corpus.')
    print('We should augment with targeted recent official-index replays before final policy promotion.')
print('\nREADY: save this notebook version/output and send public_warehouse_summary.json + public_corpus_schema.csv.')
